<a href="https://colab.research.google.com/github/gnoejh/ict1022/blob/main/Transformer/12_transformer_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

# Transformer

### 1. Using nn.Transformer for the Full Model


We will implement the full Transformer model using PyTorch's built-in `nn.Transformer`, which includes encoder and decoder layers optimized for Transformer architecture. This approach is more efficient and avoids the need to implement individual layers manually.

**Explanation**:
- **nn.Transformer**: Includes all the core components of a Transformer model, including multi-head attention, encoder and decoder layers, and residual connections.
- **nn.Embedding and Positional Encoding**: We use separate embedding and positional encoding modules, as these are not directly included in `nn.Transformer`.

This model can be used for tasks like translation, summarization, and other sequence-to-sequence tasks.
    

In [2]:

class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, embed_size)
        for pos in range(max_len):
            for i in range(0, embed_size, 2):
                pe[pos, i] = math.sin(pos / (10000 ** ((2 * i) / embed_size)))
                pe[pos, i + 1] = math.cos(pos / (10000 ** ((2 * i) / embed_size)))
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, embed_size, num_heads, num_layers, forward_expansion, dropout, max_length):
        super(TransformerModel, self).__init__()
        self.src_embedding = nn.Embedding(src_vocab_size, embed_size)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, embed_size)
        self.positional_encoding = PositionalEncoding(embed_size, max_len=max_length)
        
        # Transformer module with encoder and decoder layers
        self.transformer = nn.Transformer(
            d_model=embed_size, 
            nhead=num_heads, 
            num_encoder_layers=num_layers, 
            num_decoder_layers=num_layers, 
            dim_feedforward=forward_expansion * embed_size, 
            dropout=dropout,
            batch_first=True
        )
        
        self.fc_out = nn.Linear(embed_size, tgt_vocab_size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # Embedding and positional encoding for source and target
        src_embedded = self.positional_encoding(self.src_embedding(src))
        tgt_embedded = self.positional_encoding(self.tgt_embedding(tgt))
        
        # Transformer forward pass
        transformer_output = self.transformer(
            src_embedded, tgt_embedded, src_key_padding_mask=src_mask, tgt_key_padding_mask=tgt_mask
        )
        
        # Final output layer for token predictions
        out = self.fc_out(transformer_output)
        return out

# Example usage
src_vocab_size = 10000
tgt_vocab_size = 10000
embed_size = 16
num_heads = 4
num_layers = 3
forward_expansion = 4
dropout = 0.1
max_length = 100

transformer = TransformerModel(src_vocab_size, tgt_vocab_size, embed_size, num_heads, num_layers, forward_expansion, dropout, max_length)
src = torch.randint(0, src_vocab_size, (2, 5))  # Batch size = 2, Sequence length = 5
print("Source Input:", src)
tgt = torch.randint(0, tgt_vocab_size, (2, 5))
print("Target Input:", tgt)

output = transformer(src, tgt)
print("Transformer Model Output (logits):", output)
    

Source Input: tensor([[1148, 3049, 3705, 7337, 3493],
        [3698, 9033, 6303, 1743, 1690]])
Target Input: tensor([[  83,   15, 1126, 8644, 2730],
        [1297, 3171, 5203, 7239, 5291]])
Transformer Model Output (logits): tensor([[[-0.8460,  0.3675, -0.2098,  ...,  0.2069,  0.6686,  0.3980],
         [-0.5243, -0.0183,  0.1819,  ...,  0.4844,  0.5887, -0.6350],
         [-0.7039,  0.5401, -0.3931,  ..., -0.0588,  0.7864,  0.4394],
         [-0.9941,  0.5382, -0.3212,  ..., -0.1666,  0.1787,  0.0694],
         [-0.8014,  0.2704, -0.3773,  ..., -0.0320,  0.9234,  0.1663]],

        [[-0.8945,  0.4345,  0.1690,  ..., -0.1370,  0.2116,  0.0774],
         [-1.0265,  0.6919, -0.3169,  ...,  0.0836,  0.4904,  0.6183],
         [-0.6486,  1.1215, -0.3517,  ...,  0.4165,  0.3371,  0.1070],
         [-0.9080,  0.6453, -0.2715,  ...,  0.0419,  0.5043,  0.1621],
         [-0.7573,  0.7469, -0.4187,  ...,  0.0069,  0.7253,  0.3966]]],
       grad_fn=<ViewBackward0>)


### 2. Training Loop Example


**Objective**: Demonstrate a basic training loop with a forward and backward pass, typically used to train the Transformer on sequence-to-sequence tasks.

**Explanation**:
   - **Loss Calculation**: The model output is compared to the target sequence to compute cross-entropy loss.
   - **Backpropagation**: The loss is used to adjust model weights through gradient descent.
   - **Optimization**: Each batch adjusts model parameters to improve predictions.
    

In [3]:
# Example training loop
num_epochs = 10
learning_rate = 0.001
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Assuming padding index is 0
optimizer = optim.Adam(transformer.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    transformer.train()
    optimizer.zero_grad()

    output = transformer(src, tgt[:, :-1])  # Input all tokens except the last one
    # [batch_size, sequence_length, vocab_size]
    output = output.reshape(
        -1, output.shape[2]
    )  # [batch_size * sequence_length, vocab_size]
    tgt_output = tgt[:, 1:].reshape(-1)  # Expected output sequence shifted by one
    # tgt[:, 1:]: The target sequence, excluding the first token. 
    # This is done because the model's prediction at each time step should be compared to the next token in the sequence.
    # tgt[:, 1:].reshape(-1): Reshapes the target sequence to a 1D tensor of shape [batch_size * sequence_length]. 
    # This flattening aligns with the reshaped model output for loss computation.
    
    loss = criterion(output, tgt_output)
    loss.backward()
    optimizer.step()

    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [1/10], Loss: 9.2636
Epoch [2/10], Loss: 8.9831
Epoch [3/10], Loss: 8.8595
Epoch [4/10], Loss: 8.9279
Epoch [5/10], Loss: 8.7714
Epoch [6/10], Loss: 8.7191
Epoch [7/10], Loss: 8.6078
Epoch [8/10], Loss: 8.8084
Epoch [9/10], Loss: 8.6738
Epoch [10/10], Loss: 8.5681
